<a href="https://colab.research.google.com/github/Lynna5639582/Revision-Templates/blob/main/Classic_ML_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Logistic Regression, Decision Tree, Random Forest, GBDT

Load & Inspect Data

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()
df = pd.read_csv('YOUR_FILE.csv')  # <-- EDIT: filename

print('Shape:', df.shape)
print(df.head())
print('\nMissing values:\n', df.isna().sum())
print('\nTarget counts:\n', df['YOUR_TARGET'].value_counts())  # <-- EDIT: target column

Clean & Encode: Only run what data needs

In [ ]:
# 1. Drop useless ID columns or columns with >30% missing values
df = df.drop(columns=['id', 'dataset', 'ca', 'thal', 'slope'], errors='ignore')
df = df.dropna()

# 2. Separate target BEFORE encoding so y stays untouched
target_col = 'num'  # <-- ONLY EDIT THIS NAME IN THE EXAM
y = df[target_col]
X = df.drop(columns=[target_col])

# 3. Automatically convert ALL text and True/False columns into 0s and 1s
X = pd.get_dummies(X, drop_first=True, dtype=int)

print("Ready for Block 3! X shape:", X.shape)

Split & Scale

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Ensure all text columns in X are encoded into numbers first
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# Split using the encoded dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=4567, stratify=y
)

# Scale the features using training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully! Shape of scaled train set:", X_train_scaled.shape)

Train & Compare Models (Train vs. Test Table)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=4567),
    'Decision Tree': DecisionTreeClassifier(random_state=4567),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=4567, n_jobs=-1),
    'GBDT': GradientBoostingClassifier(n_estimators=80, random_state=4567),
}

results = []
for name, model in models.items():
    train_X = X_train_scaled if name == 'Logistic Regression' else X_train
    test_X  = X_test_scaled  if name == 'Logistic Regression' else X_test

    model.fit(train_X, y_train)

    for split, features, target in [('Train', train_X, y_train), ('Test', test_X, y_test)]:
        pred = model.predict(features)
        precision, recall, f1, _ = precision_recall_fscore_support(
            target, pred, average='macro', zero_division=0
        )
        results.append({
            'Model': name, 'Split': split,
            'Accuracy': accuracy_score(target, pred),
            'Macro Precision': precision,
            'Macro Recall': recall,
            'Macro F1': f1
        })

results_df = pd.DataFrame(results)
display(results_df.round(3))

Confusion Matrix & Feature Importance

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model = models['Random Forest']  # <-- EDIT: pick winning model
test_features = X_test_scaled if best_model == models['Logistic Regression'] else X_test

ConfusionMatrixDisplay.from_predictions(y_test, best_model.predict(test_features))
plt.title("Confusion Matrix - Best Model")
plt.show()